In [ ]:
import sys
print(sys.executable)
import torch
import torch.nn as nn
print(torch.__version__)
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your     (x^1)
     [0.55, 0.87, 0.66], # journey  (x^2)
     [0.57, 0.85, 0.64], # starts   (x^3)
     [0.22, 0.58, 0.33], # with     (x^4)
     [0.77, 0.25, 0.10], # one      (x^5)
     [0.05, 0.80, 0.55]] # step     (x^6)
)

c:\Users\johan\anaconda3\python.exe
2.13.0+cpu


Beispieldaten aus dem buch Kapitel 3.3

In [2]:
print(inputs.shape) # torch.Size([6, 3])
print(inputs.shape[0])

torch.Size([6, 3])
6


In [3]:
def att_score(query):
    attn_scores = torch.empty(query.shape[0], query.shape[0])
    for i , q in enumerate(query):
        for j, k in enumerate(query):
            attn_scores[i,j] = torch.dot(q, k)
    return attn_scores

In [ ]:
attention_scores = att_score(inputs)
print(attention_scores)
att_scores = inputs @ inputs.T
print(att_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
def softmax(tensor):
    exp_tensor = torch.exp(tensor)
    sum_exp = torch.sum(exp_tensor, dim=1, keepdim=True)
    return exp_tensor / sum_exp


In [ ]:
weights = softmax(attention_scores)
att_weights = torch.softmax(att_scores, dim=-1)
print(weights)
print(att_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [ ]:
def context_vector(weights, inputs):
    context_vectors = torch.empty(weights.shape[0], inputs.shape[1])
    for i in range(weights.shape[0]):
        context_vectors[i] = torch.sum(weights[i].unsqueeze(1) * inputs, dim=0)
        # context_vectors += weights[i] * inputs
    return context_vectors

context_vectors = context_vector(weights, inputs)
print(context_vectors)

context_vectors = weights @ inputs
print(context_vectors)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, input_dim, d_out, qkv_bias=False):
        super().__init__()

        self.d_out = d_out
        # Define the linear layers for query, key, and value
        self.w_query = nn.Linear(input_dim, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(input_dim, d_out, bias=qkv_bias)
        self.w_value = nn.Linear(input_dim, d_out, bias=qkv_bias)

    def forward(self, inputs):
        # Compute query, key, and value matrices
        query = self.w_query(inputs)
        key = self.w_key(inputs)
        value = self.w_value(inputs)

        attention_scores = query @ key.transpose(-2, -1)
        attention_scores = attention_scores / (key.shape[-1] ** 0.5)  # Scale the scores
        attention_weights = torch.softmax(attention_scores, dim=-1)
        context = attention_weights @ value

        return context

In [ ]:
class SelfAttention_raw(nn.Module):
    def __init__(self,input_dim ,d_out):
        super().__init__()

        self.d_out = d_out
        # Define the linear layers for query, key, and value

        self.w_query = nn.Parameter(torch.randn(input_dim, d_out))
        self.w_key = nn.Parameter(torch.randn(input_dim, d_out))
        self.w_value = nn.Parameter(torch.randn(input_dim, d_out))

    def forward(self, inputs):
        # Compute query, key, and value matrices
        query = inputs @ self.w_query
        key = inputs @ self.w_key
        value = inputs @ self.w_value

        attention_scores = query @ key.transpose(-2, -1)
        attention_scores = attention_scores / (key.shape[-1] ** 0.5)  # Scale the scores
        attention_weights = torch.softmax(attention_scores, dim=-1)
        context = attention_weights @ value

        return context